In [0]:
-- Drop the old table first (it has different column names), so this cell is always re-runnable
-- DROP TABLE IF EXISTS workspace.default.sales;

-- CREATE TABLE workspace.default.sales (
--   productid   INT,
--   productname STRING,
--   sales       DECIMAL(12,2),
--   orderStatus STRING,
--   orderDate   DATE
-- );

-- Insert sample data, including repeated (duplicate) rows.
-- Delta tables keep repeated rows by default — there is no PRIMARY KEY
-- or UNIQUE constraint that would remove them.
-- TRUNCATE TABLE workspace.default.sales;

-- INSERT INTO workspace.default.sales (productid, productname, sales, orderStatus, orderDate) VALUES
--   (101, 'Laptop',      1200.00, 'shipped', '2022-01-01'),
--   (102, 'Mouse',          25.50, 'delivered', '2022-01-02'),
--   (103, 'Keyboard',       75.00, 'shipped', '2022-02-03'),
--   (104, 'Headphones',  150.00, 'pending', '2022-03-04'),
--   (105, 'Monitor',     300.00, 'shipped', '2022-01-04'),  
--   (106, 'Monitor',     500.00, 'delivered', '2022-04-06'),
--   (101, 'Laptop',       1200.00, 'shipped', '2022-02-03'),
--   (102, 'Mouse',          25.50, 'shipped', '2022-01-04'),
--   (104, 'Headphones',  150.00, 'shipped', '2022-05-08'),
--   (102, 'Mouse',          25.50, 'delivered', '2022-02-04'),
--   (101, 'Laptop',       1200.00, 'shipped', '2022-02-05'),
--   (104, 'Headphones',  150.00, 'delivered', '2022-01-04'),
--   (105, 'Monitor',     300.00, 'shipped', '2022-03-05'),
--   (106, 'Monitor',     500.00, 'shipped', '2022-04-04');

SELECT *
FROM workspace.default.sales;

-- total sales by product
SELECT  
    productname,
    sum(sales) as total_sales
FROM workspace.default.sales
GROUP BY productname;

SELECT  
    productid,
    productname,
    sum(sales) as total_sales
FROM workspace.default.sales
GROUP BY productname, productid;

-- Group by has certain limitation and cannot be used for advanced aggregation and ranking that's why we need to use window function
-- Window function is a function that operates on a set of rows that are somehow related to the current row. Window functions are used to perform calculations across a set of rows that are somehow related to the current row..

-- WINDOW FUNCTION
/* An SQL window function performs a calculation across a set of table rows that are somehow related to the current row—similar to an aggregate function, except it does not collapse the rows into a single group. Each row retains its individual identity in the output.

The core difference is that GROUP BY collapses multiple rows into a single summary row per group, whereas window functions compute calculations across related rows while keeping every original row intact 


    WINDOW_FUNCTION()
    OVER (
        PARTITION BY column_name
        ORDER BY column_name
        ROWS BETWEEN frame_start AND frame_end)
*/
SELECT
    productid,
    productname,
    orderStatus,
    sales,
    sum(sales) OVER () AS totalSales,
    sum(sales) OVER (PARTITION BY productname) AS SalesByProduct,
    sum(sales) OVER (PARTITION BY productname, orderStatus) AS SalesByPeoductAndStatus
FROM workspace.default.sales
ORDER BY productid;

-- ORDER BY Clause
-- partition by month and order by sales
SELECT
    productid,
    productname,
    orderStatus,
    date_format(orderDate, 'MMMM') AS month,
    sales,
    sum(sales) OVER (PARTITION BY month(orderDate) ORDER BY sales DESC) AS SalesByMonth
FROM workspace.default.sales
ORDER BY productid;

-- RANK()
SELECT 
    productid,
    productname,
    sales,
    rank() OVER (ORDER BY sales) AS rank,
    dense_rank() OVER (ORDER BY sales) AS desnse_rank,
    row_number() OVER (ORDER BY sales) AS row_number
FROM workspace.default.sales;


SELECT 
    productname,
    sales,
    month(orderDate) AS month,
    SUM(sales) OVER(PARTITION BY month(orderDate)) AS sales_by_month
FROM workspace.default.sales;


SELECT 
    productname,
    sales,
    month(orderDate) AS month,
    SUM(sales) OVER(PARTITION BY month(orderDate)) AS sales_by_month,
    SUM(sales) OVER(ORDER BY month(orderDate)) AS orderMonth
FROM workspace.default.sales;

-- FRAME Clause
SELECT
    productid,
    orderDate,
    orderStatus,
    sales,
    sum(sales) OVER(PARTITION BY orderStatus ORDER BY orderDate
    ROWS BETWEEN CURRENT ROW AND 2 FOLLOWING) totalSales
FROM workspace.default.sales;


SELECT
    productid,
    orderDate,
    orderStatus,
    sales,
    sum(sales) OVER(PARTITION BY orderStatus ORDER BY orderDate
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW ) totalSales,
    sum(sales) OVER(PARTITION BY orderStatus ORDER BY orderDate) defaultFrame
FROM workspace.default.sales;

--- #####################################
--    RULES FOR USING WINDOW FUNCTION
--  ####################################
/*
        1. Window func can only be used with SELECT and ORDERBY
        2. Nesting of window func is not allowed
        3. SQL execute Window func after the WHERE clause
        4. window func with groupBy is used only when the same columns are used
 */


-- Rank product based on their total sales
SELECT  
    productid,
    sum(sales) AS totalSales,
    rank() over(ORDER BY sum(sales) DESC) rankCustomers
FROM workspace.default.sales
GROUP BY productid;

-- window functions cannot be nested: compute the per-product total in a subquery, then rank over it
SELECT DISTINCT
    productid,
    totalSales,
    rank() OVER (ORDER BY totalSales DESC) AS rankCustomers
FROM (
    SELECT
        productid,
        sum(sales) OVER (PARTITION BY productid) AS totalSales
    FROM workspace.default.sales
);


-- agggregate Window function
SELECT
    productname,
    count(sales) OVER(PARTITION BY productname) as countByProduct
FROM workspace.default.sales;

-- total number of orders
-- additionally provide id and order date
-- total number of orders for each product
SELECT
    productid,
    orderDate,
    productname,
    count(*) OVER(PARTITION BY productname) AS productCOUNT,
    count(*) OVER() AS totalOrder
FROM workspace.default.sales;


-- find total number of customers in customers table
SELECT
    first_name,
    count(*) OVER() AS totalCustomers,
    dense_rank() OVER(ORDER BY first_name) AS rank,
    len(first_name) AS length,
    count(score) OVER() AS totalNumberOfScore
FROM workspace.default.customers;

-- checking DUPLICSTES
SELECT *
FROM (
    SELECT 
    productid,
    count(*) OVER(PARTITION BY productid) AS pk_check
FROM workspace.default.sales
)t WHERE pk_check > 1;

-- total sales across all orders
--total sales for eac product
SELECT
    productname,
    sum(sales) OVER() totalSales,
    sum(sales) OVER(PARTITION BY productname) as toalSalesByProduct
FROM workspace.default.sales;

-- find the percentage contribution of each product to total sales
SELECT
    productname,
    sales,
    sum(sales) OVER() totalSales,
    round(sales / totalSales * 100, 2) AS percentageSales,
    avg(coalesce(sales, 0)) OVER(PARTITION BY productname) AS averageSales
FROM workspace.default.sales
ORDER BY percentageSales DESC;

-- find the product where sales are hugher than average sales
SELECT *
FROM (
    SELECT
    productid,
    productname,
    sales,
    avg(sales) OVER() averageSales
FROM workspace.default.sales
) WHERE sales > averageSales;

-- minimum and maximum sales by product
SELECT
    productname,
    sales,
    min(sales) OVER(PARTITION BY productname ORDER by productname) minimumSale,
    max(sales) OVER(PARTITION BY productname) maximumSale
FROM workspace.default.sales;

-- running total
SELECT 
    date_format(orderDate, 'MMMM') AS month,
    sales,
    sum(sales) OVER(
    PARTITION BY month(orderdate)
    ) totalSalesByMonth
FROM workspace.default.sales;

-- rolling totoal
SELECT 
    date_format(orderDate, 'MMMM') AS month,
    sales,
    sum(sales) OVER(
    ORDER BY month(orderdate)
    ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ) rollingTotalForThreeMonths
FROM workspace.default.sales;

-- rolling totoal
SELECT 
    salesMonth AS month,
    sales,
    sum(sales) OVER(
    ORDER BY monthNum
    ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ) rollingTotalForThreeMonths
FROM (
    SELECT 
        date_format(orderDate, 'MMMM') AS salesMonth,
        month(orderDate) AS monthNum,
        sum(sales) AS sales
    FROM workspace.default.sales
    GROUP BY salesMonth, monthNum
);

-- ##########################################
--                 RANKING
-- ##########################################

/*
Integer-Based Functions: ROW_NUMBER vs RANK vs DENSE_RANK
------------------------------------------------------------------
Imagine you have three employees with the exact same salary ($10,000), competing for the top spots:
• ROW_NUMBER(): Strictly numbers rows 1, 2, 3, 4. If rows 1, 2, and 3 have identical values, it arbitrarily numbers them anyway. It is ideal for pagination or finding a single "most recent" record.
• RANK(): Standard sports leaderboard logic. The three tying employees all get rank 1. Because three people took up the top spots, the next employee gets rank 4 (1, 1, 1, 4).
• DENSE_RANK(): No gaps. The three tying employees get rank 1. The next employee gets rank 2 (1, 1, 1, 2). It is ideal when you need to find the "3rd highest distinct salary" regardless of how many people share the top spots.

2. Percentage-Based Functions: CUME_DIST vs PERCENT_RANK
---------------------------------------------------------------------
These functions evaluate relative position within a data distribution rather than assigning absolute integers.
• CUME_DIST() (Cumulative Distribution):
	• Formula: {position_number / total_row}
	• Behavior: It tells you what fraction of the data falls at or below the current record. The very first row will always be greater than 0, and the final row will always be exactly 1.0.
• PERCENT_RANK() (Percentile Rank):
	• Formula: {position_number - 1 / total_ro - 1}
	• Behavior: It mirrors the behavior of RANK() but scales it between 0 and 1. The very first row in the dataset will always have a score of 0

    CUME_DIST PRIORITIZES THE LAST ROW (REACHING EXACTLY 1.0 FOR THE FINAL VALUE), WHILE PERCENT_RANK PRIORITIZES THE FIRST ROW (STARTING AT EXACTLY 0.0 FOR THE INITIAL VALUE
 */

SELECT
    productname,
    sales,
    row_number() OVER(ORDER BY sales DESC) AS topSales,
    rank() OVER(ORDER BY sales DESC) AS rank,
    dense_rank() OVER(ORDER BY sales DESC) as denseRank,
    percent_rank() OVER(ORDER BY sales DESC) as perctRank,
    cume_dist()  OVER(ORDER BY sales DESC) AS cumeDist
FROM workspace.default.sales;

-- 3rd and 4th highest sales of a product
SELECT
    DISTINCT productname AS fourthAndThirdHighestSaledProduct,
    sales
FROM(
    SELECT
        productname,
        sales,
        dense_rank() OVER(ORDER BY sales DESC) as denseRank
    FROM workspace.default.sales
)t WHERE denserank in (4, 3);

-- top highest saless for each product
SELECT 
    productname,
    sales
FROM (
    SELECT
        productname,
        sales,
        row_number() OVER(PARTITION BY productname ORDER BY sales DESC) AS highestPerProduct
    FROM workspace.default.sales
)t WHERE highestperproduct = 1;

-- NTILE
-- divide the row into specified number of approximately equal group
-- PAGINATION
/* Pagination in SQL is the process of dividing a large dataset into smaller, discrete chunks (pages) to optimize application performance and improve user experience. There are two main strategies used to implement pagination: Offset-Based Pagination and Cursor-Based (Keyset) Pagination. */
SELECT
    productname,
    ntile(5) OVER(ORDER BY sales DESC) as groups
FROM workspace.default.sales;

-- VALUE window function
-- we have to provide frame for last_value
SELECT
    month, 
    sales,
    first_value(sales) OVER(ORDER BY monthNum) as firstValue,
    last_value(sales) OVER(ORDER BY monthNum
                            ROWS BETWEEN CURRENT ROW AND UNBOUNDED FOLLOWING) as lastvalue,
    lead(sales, 1, 100) OVER(ORDER BY monthnum) as lead,
    lag(sales) OVER(ORDER BY monthNUM) as lag
 FROM (
    SELECT  
        month(orderDate) as monthNum,
        date_format(orderDate, 'MMMM') as month,
        sum(sales) AS sales
    FROM workspace.default.sales
    GROUP BY month(orderDate), date_format(orderDate, 'MMMM')
);

-- MoM Analysis
SELECT
    *,
    curentmonthsales - previousmonthsales AS MoM_Change,
    round(mom_change / previousmonthsales * 100, 1) AS perct_change
FROM (
    SELECT
        month(orderdate) as orderMonth,
        sum(sales) as curentMonthSales,
        lag(sum(sales)) OVER(ORDER BY month(orderdate)) as previousMonthSales
    FROM workspace.default.sales
    GROUP BY orderMonth
)t;


